# ⚡ 06: Real-Time Event Streaming & CDC Lakehouse Engine

This notebook demonstrates the **Real-Time Streaming & Change Data Capture (CDC)** module of the Tech News Analytical Platform.

### Core Architectural Mechanics:
1. **Micro-Batch Event Stream Ingestion**: Real-time JSON stream ingestion of tech news and ARR revisions.
2. **Event-Time Watermarking**: Automatic filtering of late-arriving events based on event-time latency boundaries.
3. **Dead-Letter Queue (DLQ)**: Automatic error quarantining of malformed/corrupt records with failure telemetry.
4. **CDC Atomic `MERGE INTO` Reconciliation**: Reconciling `INSERT` (`I`), `UPDATE` (`U`), and `DELETE` (`D`) op-codes into Gold star-schema tables.
5. **Real-Time DuckDB Analytics**: Querying reconciled metrics and ARR snapshots.

In [ ]:
import os
import json
import pandas as pd
import duckdb
from medallion.streaming_cdc import CDCStreamProcessor

processor = CDCStreamProcessor()
print("[+] CDC Stream Processor initialized!")
print("    - Stream Landing Dir:", processor.stream_dir)
print("    - DLQ Quarantine Dir:", processor.dlq_dir)
print("    - Warehouse Dir:     ", processor.warehouse_dir)

## 📥 1. Generate Realistic Streaming Micro-Batches

We generate 3 micro-batches simulating live events:
- **Batch 1**: Live breaking news & ARR insertions (`op_code='I'`)
- **Batch 2**: Financial revision update (`op_code='U'`) and article retraction (`op_code='D'`)
- **Batch 3**: Late-arriving event past watermark window + corrupt schema record for DLQ

In [ ]:
batch_files = processor.generate_sample_stream_batches()
for b in batch_files:
    print(f"Generated batch: {os.path.basename(b)}")
    with open(b, 'r') as f:
        for line in f:
            print("  ->", line.strip())

## ⚙️ 2. Process Micro-Batch 1: Live Inserts (`op_code='I'`)

In [ ]:
m1 = processor.process_stream_batch(batch_files[0], watermark_hours=48.0)
print("Batch 1 Results:", json.dumps(m1, indent=2))

## 🔄 3. Process Micro-Batch 2: CDC Updates & Retractions (`op_code='U'`, `op_code='D'`)

In [ ]:
m2 = processor.process_stream_batch(batch_files[1], watermark_hours=48.0)
print("Batch 2 Results:", json.dumps(m2, indent=2))

## 🛡️ 4. Process Micro-Batch 3: Edge Cases (Late Event + DLQ Quarantine)

In [ ]:
m3 = processor.process_stream_batch(batch_files[2], watermark_hours=48.0)
print("Batch 3 Results:", json.dumps(m3, indent=2))

print("\n--- Inspecting Dead-Letter Queue (DLQ) Files ---")
for dlq_f in os.listdir(processor.dlq_dir):
    if dlq_f.endswith('.json'):
        with open(os.path.join(processor.dlq_dir, dlq_f), 'r') as fp:
            print(json.dumps(json.load(fp), indent=2))

## 📊 5. Real-Time DuckDB Analytical SQL over Reconciled Warehouse

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE TABLE dim_company AS SELECT * FROM read_csv_auto('{processor.warehouse_dir}/dim_company.csv')")
con.execute(f"CREATE TABLE fct_article AS SELECT * FROM read_csv_auto('{processor.warehouse_dir}/fct_article.csv')")
con.execute(f"CREATE TABLE fct_arr AS SELECT * FROM read_csv_auto('{processor.warehouse_dir}/fct_arr_observation.csv')")
con.execute(f"CREATE TABLE view_latest AS SELECT * FROM read_csv_auto('{processor.warehouse_dir}/view_company_latest_arr.csv')")

print("=== Top Companies by Reconciled Latest ARR (USD Millions) ===")
df_res = con.execute("""
    SELECT company_name, industry, latest_arr_usd_M, total_observations_recorded
    FROM view_latest
    WHERE latest_arr_usd_M IS NOT NULL
    ORDER BY latest_arr_usd_M DESC
    LIMIT 10
""").df()
print(df_res.to_string(index=False))